In [3]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("Redbus_Ground_Truth_Final.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip()

print(f"Jumlah baris awal: {df.shape[0]}")

# Menghapus ulasan yang tidak memiliki label pada ketiga aspek
kolom_aspek = ['Ticketing', 'Information Channels', 'Travel Experience']
df = df.dropna(subset=kolom_aspek, how='all').reset_index(drop=True)

print(f"Jumlah baris setelah filter aspek kosong: {df.shape[0]}\n")

Jumlah baris awal: 4463
Jumlah baris setelah filter aspek kosong: 1977



In [7]:
df['review_casefolding'] = df['review'].astype(str).str.lower()

display(df[['review', 'review_casefolding']].head(10))

,review,review_casefolding
0,jadwal yg tertulis di aplikasi g sesuai..kl se...,jadwal yg tertulis di aplikasi g sesuai..kl se...
1,tolong jangan pada beli tiket di sini kalau ka...,tolong jangan pada beli tiket di sini kalau ka...
2,rekomendasi dan sangat memudahkan,rekomendasi dan sangat memudahkan
3,kenapa harga tiket dan harga di aplikasi tidak...,kenapa harga tiket dan harga di aplikasi tidak...
4,Red bus saya bayar mahal tapi malah dapat kurs...,red bus saya bayar mahal tapi malah dapat kurs...
5,cara pesan tiket mudah masa kini..,cara pesan tiket mudah masa kini..
6,Aplikasi yg bagus dan simple. Sangat fast... r...,aplikasi yg bagus dan simple. sangat fast... r...
7,Sangat membantu dan harga sangat bersahabat,sangat membantu dan harga sangat bersahabat
8,Menu oke ckp jelas namun pada saat melakukan t...,menu oke ckp jelas namun pada saat melakukan t...
9,Gua udah pesan tiket udah bayar malah dicancel...,gua udah pesan tiket udah bayar malah dicancel...


In [8]:
import re

emoji_pattern = re.compile(
    "["
    u"\U0001F600-\U0001F64F" u"\U0001F300-\U0001F5FF"
    u"\U0001F680-\U0001F6FF" u"\U0001F1E0-\U0001F1FF"
    u"\U00002700-\U000027BF" u"\U0001F900-\U0001F9FF"
    u"\U0001FA70-\U0001FAFF" u"\U00002500-\U00002BEF"
    u"\U0001F100-\U0001F1FF"
    "]+", flags=re.UNICODE
)

def clean_text(text):
    # Validasi input (Teks sudah dalam bentuk lowercase dari tahap case folding)
    if pd.isna(text) or text == 'nan':
        return ''
    
    # Menghapus elemen non-tekstual dan tanda baca
    text = re.sub(r'@[a-z0-9_]+', '', text)     # Hapus mention
    text = re.sub(r'#[a-z0-9_]+', '', text)     # Hapus hashtag
    text = re.sub(r'rt[\s]+', '', text)         # Hapus indikator retweet
    text = re.sub(r"http\S+", '', text)         # Hapus tautan/URL
    text = re.sub(r'[0-9]+', '', text)          # Hapus karakter numerik
    text = text.replace('\n', ' ')              # Ganti baris baru dengan spasi
    text = emoji_pattern.sub(r'', text)         # Hapus karakter emoji
    text = re.sub(r'[^\w\s]', '', text)         # Hapus seluruh tanda baca
    text = re.sub(r'\s+', ' ', text).strip()    # Hapus spasi ganda/berlebih di awal dan akhir
    
    return text

# Menerapkan fungsi cleansing dari hasil kolom case folding
df['review_clean'] = df['review_casefolding'].apply(clean_text)

# Menampilkan komparasi hasil cleansing
display(df[['review_casefolding', 'review_clean']].head(20))

,review_casefolding,review_clean
0,jadwal yg tertulis di aplikasi g sesuai..kl se...,jadwal yg tertulis di aplikasi g sesuaikl seli...
1,tolong jangan pada beli tiket di sini kalau ka...,tolong jangan pada beli tiket di sini kalau ka...
2,rekomendasi dan sangat memudahkan,rekomendasi dan sangat memudahkan
3,kenapa harga tiket dan harga di aplikasi tidak...,kenapa harga tiket dan harga di aplikasi tidak...
4,red bus saya bayar mahal tapi malah dapat kurs...,red bus saya bayar mahal tapi malah dapat kurs...
5,cara pesan tiket mudah masa kini..,cara pesan tiket mudah masa kini
6,aplikasi yg bagus dan simple. sangat fast... r...,aplikasi yg bagus dan simple sangat fast respo...
7,sangat membantu dan harga sangat bersahabat,sangat membantu dan harga sangat bersahabat
8,menu oke ckp jelas namun pada saat melakukan t...,menu oke ckp jelas namun pada saat melakukan t...
9,gua udah pesan tiket udah bayar malah dicancel...,gua udah pesan tiket udah bayar malah dicancel...


In [11]:
kamus = pd.read_csv('colloquial-indonesian-lexicon.csv')
normalization_dict = dict(zip(kamus['slang'].str.lower(), kamus['formal'].str.lower()))

def normalize_text(text):
    if not text:
        return ''
    words = text.split()
    normalized_words = [normalization_dict.get(word, word) for word in words]
    return ' '.join(normalized_words)

df['review_normalized'] = df['review_clean'].apply(normalize_text)
display(df[['review_clean', 'review_normalized']].head(25))

,review_clean,review_normalized
0,jadwal yg tertulis di aplikasi g sesuaikl seli...,jadwal yang tertulis di aplikasi enggak sesuai...
1,tolong jangan pada beli tiket di sini kalau ka...,tolong jangan pada beli tiket di sini kalau ka...
2,rekomendasi dan sangat memudahkan,rekomendasi dan sangat memudahkan
3,kenapa harga tiket dan harga di aplikasi tidak...,kenapa harga tiket dan harga di aplikasi tidak...
4,red bus saya bayar mahal tapi malah dapat kurs...,red bus saya bayar mahal tapi malah dapat kurs...
5,cara pesan tiket mudah masa kini,cara pesan tiket mudah masa kini
6,aplikasi yg bagus dan simple sangat fast respo...,aplikasi yang bagus dan simple sangat fast res...
7,sangat membantu dan harga sangat bersahabat,sangat membantu dan harga sangat bersahabat
8,menu oke ckp jelas namun pada saat melakukan t...,menu oke cukup jelas namun pada saat melakukan...
9,gua udah pesan tiket udah bayar malah dicancel...,gua sudah pesan tiket sudah bayar malah dicanc...


In [6]:
from deep_translator import GoogleTranslator
import time
from tqdm import tqdm
import pandas as pd
from IPython.display import display

# Inisialisasi Translator
translator_to_en = GoogleTranslator(source='id', target='en')
translator_to_id = GoogleTranslator(source='en', target='id')

# Definisi Fungsi Back-Translation
def back_translate(text):
    """
    Fungsi untuk melakukan terjemahan dari Bahasa Indonesia (id) 
    ke Bahasa Inggris (en), lalu kembali ke Bahasa Indonesia (id).
    """
    if not isinstance(text, str) or text.strip() == '':
        return text
    
    try:
        # Tahap 1: Terjemahkan ID -> EN
        translated = translator_to_en.translate(text)
        
        # Tahap 2: Terjemahkan EN -> ID
        back_translated = translator_to_id.translate(translated)
        
        return back_translated
    except Exception as e:
        return text

# Eksekusi pada Seluruh Data dengan Progress Bar
print("[PROSES] Memulai Back-Translation pada seluruh data...")
print("Catatan: Proses ini memakan waktu beberapa menit.")

# Mengaktifkan ekstensi progress bar untuk pandas
tqdm.pandas(desc="Proses Back-Translation")

# Menerapkan fungsi ke seluruh baris dataset
df['review_backtranslated'] = df['review_normalized'].progress_apply(back_translate)

print("\n[SELESAI] Back-Translation.")

# Tampilkan Hasil dan Simpan
display(df[['review_normalized', 'review_backtranslated']].head(10))

# Menyimpan hasil ke file CSV agar aman
nama_file_output = "hasil_back_translation_redbus_final.csv"
df.to_csv(nama_file_output, index=False, encoding="utf-8-sig")

print(f"Dataset terbaru telah berhasil disimpan ke: {nama_file_output}")

[PROSES] Memulai Back-Translation pada seluruh data...
Catatan: Proses ini memakan waktu beberapa menit.


Proses Back-Translation: 100%|██████████| 1977/1977 [49:59<00:00,  1.52s/it] 


[SELESAI] Back-Translation.


,review_normalized,review_backtranslated
0,jadwal yang tertulis di aplikasi enggak sesuai...,jadwal yang tertulis di aplikasi tidak sesuai ...
1,tolong jangan pada beli tiket di sini kalau ka...,Mohon jangan membeli tiket di sini jika tidak ...
2,rekomendasi dan sangat memudahkan,rekomendasi dan sangat mudah
3,kenapa harga tiket dan harga di aplikasi tidak...,kenapa harga tiket dan harga di aplikasi tidak...
4,red bus saya bayar mahal tapi malah dapat kurs...,Saya membayar banyak untuk bus merah tetapi sa...
5,cara pesan tiket mudah masa kini,cara mudah hari ini untuk memesan tiket
6,aplikasi yang bagus dan simple sangat fast res...,"aplikasi bagus dan sederhana, respon sangat ce..."
7,sangat membantu dan harga sangat bersahabat,sangat membantu dan harga sangat bersahabat
8,menu oke cukup jelas namun pada saat melakukan...,"Oke menunya cukup jelas, namun saat melakukan ..."
9,gua sudah pesan tiket sudah bayar malah dicanc...,"Saya pesan tiket dan bayar, tapi dibatalkan, m..."


Dataset terbaru telah berhasil disimpan ke: hasil_back_translation_redbus_final.csv
